# YOLOv8 1-Class Traffic Sign Detection Training

This notebook downloads your custom Kaggle dataset using the Kaggle API (kaggle.json), extracts the raw GTSDB dataset, preprocesses it into a 1-class YOLOv8 format, and trains a YOLOv8n detector. This runs entirely in Google Colab or Kaggle without uploading local data.

### Step 1: Install Dependencies

In [ ]:
!pip install ultralytics

### Step 2: Upload your Kaggle API credentials (kaggle.json)
Run this cell and upload the `kaggle.json` file downloaded from your Kaggle Account Settings.

In [ ]:
from google.colab import files
files.upload() # Select your kaggle.json file

### Step 3: Configure Kaggle API and Download Dataset
Move your credentials to the correct location and download your custom Kaggle dataset containing the raw zips.

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d hanuma2048/trafficsense-raw-data

### Step 4: Extract Dataset and Clean Up Zips
Extract the zip downloaded from Kaggle. Note that Kaggle automatically unzips uploaded files, but in case they are nested as zips, we support extracting them as well. Finally, we remove zip files to save workspace memory.

In [ ]:
import zipfile
from pathlib import Path

kaggle_zip = Path("trafficsense-raw-data.zip") 

# Extract Kaggle zip bundle
if kaggle_zip.exists():
    print("Extracting Kaggle dataset bundle...")
    with zipfile.ZipFile(kaggle_zip, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Kaggle bundle extraction complete.")
    kaggle_zip.unlink()

# Fallback check: If the bundle contained raw zips instead of extracted folders
gtsdb_zip = Path("FullIJCNN2013.zip")
if gtsdb_zip.exists():
    print("Found nested FullIJCNN2013.zip, extracting...")
    with zipfile.ZipFile(gtsdb_zip, 'r') as zip_ref:
        zip_ref.extractall("GTSDB_extracted")
    gtsdb_zip.unlink()

### Step 5: Preprocess GTSDB for 1-Class YOLOv8
We search the entire workspace dynamically for `gt.txt` to find where the GTSDB images and labels are located. We parse the annotations and map all classes to `0` (`traffic_sign`). We also split the data into training (80%) and validation (20%) sets and convert `.ppm` files to `.jpg`.

In [ ]:
import random
import shutil
from PIL import Image
from pathlib import Path

# Search the entire workspace dynamically for gt.txt
gt_candidates = list(Path(".").glob("**/gt.txt"))
if len(gt_candidates) > 0:
    gt_file = gt_candidates[0]
    gtsdb_src = gt_file.parent
    print(f"Found annotations file at: {gt_file}")
    print(f"Source images directory: {gtsdb_src}")
else:
    raise FileNotFoundError("Could not locate gt.txt annotations file anywhere in the workspace.")

yolo_dir = Path("yolo_data")

# Setup folders
splits = ["train", "valid"]
for split in splits:
    (yolo_dir / split / "images").mkdir(parents=True, exist_ok=True)
    (yolo_dir / split / "labels").mkdir(parents=True, exist_ok=True)

# Helper function to convert bounding box coordinates to YOLO format
def convert_bbox_to_yolo(x1, y1, x2, y2, img_w, img_h):
    dw = 1.0 / img_w
    dh = 1.0 / img_h
    xc = (x1 + x2) / 2.0
    yc = (y1 + y2) / 2.0
    w = x2 - x1
    h = y2 - y1
    return xc * dw, yc * dh, w * dw, h * dh

# Read annotations
annotations = {}
with open(gt_file, "r") as f:
    for line in f:
        line = line.strip()
        if not line: continue
        parts = line.split(";")
        if len(parts) < 6: continue
        filename = parts[0]
        x1, y1, x2, y2 = map(float, parts[1:5])
        if filename not in annotations:
            annotations[filename] = []
        annotations[filename].append((x1, y1, x2, y2))

# Split into Train and Validation
all_ppm = list(gtsdb_src.glob("*.ppm"))
random.seed(42)
random.shuffle(all_ppm)
split_idx = int(len(all_ppm) * 0.8)

image_splits = {
    "train": all_ppm[:split_idx],
    "valid": all_ppm[split_idx:]
}

print("Preprocessing images and labels...")
for split, images in image_splits.items():
    for ppm_path in images:
        filename = ppm_path.name
        jpg_filename = ppm_path.stem + ".jpg"
        
        dst_img_path = yolo_dir / split / "images" / jpg_filename
        dst_lbl_path = yolo_dir / split / "labels" / (ppm_path.stem + ".txt")
        
        # Convert image to JPG
        with Image.open(ppm_path) as img:
            img_w, img_h = img.size
            img.convert("RGB").save(dst_img_path, "JPEG")
            
        # Write labels (collapse classes to 0)
        if filename in annotations:
            with open(dst_lbl_path, "w") as out_f:
                for (x1, y1, x2, y2) in annotations[filename]:
                    xc, yc, w, h = convert_bbox_to_yolo(x1, y1, x2, y2, img_w, img_h)
                    out_f.write(f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")
        else:
            open(dst_lbl_path, "w").close()
            
print("Preprocessing complete.")

### Step 6: Generate data.yaml configuration file

In [ ]:
import yaml

data_yaml_content = {
    'path': str(yolo_dir.absolute()),
    'train': 'train/images',
    'val': 'valid/images',
    'nc': 1,
    'names': {
        0: 'traffic_sign'
    }
}

with open('data.yaml', 'w') as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False)
print("data.yaml generated.")

### Step 7: Train the YOLOv8 Detector
We train YOLOv8n for 50 epochs using GPU acceleration.

In [ ]:
from ultralytics import YOLO

# Initialize YOLOv8n
model = YOLO('yolov8n.pt')

# Start training
results = model.train(
    data='data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0 # Uses the primary GPU
)

### Step 8: Visualize Training Performance
We read and display the training performance curves (losses, precision, recall) and validation confusion matrix generated by YOLOv8.

In [ ]:
from IPython.display import Image, display
import os

results_path = 'runs/detect/train/results.png'
cm_path = 'runs/detect/train/confusion_matrix.png'

if os.path.exists(results_path):
    print("Training Loss and Metric Curves:")
    display(Image(results_path, width=800))

if os.path.exists(cm_path):
    print("Validation Confusion Matrix:")
    display(Image(cm_path, width=600))

# Run and print numeric validation summary
print("Running validation evaluation...")
metrics = model.val()
print(f"mAP@0.5: {metrics.results_dict['metrics/mAP50(B)']:.4f}")
print(f"mAP@0.5:0.95: {metrics.results_dict['metrics/mAP50-95(B)']:.4f}")

### Step 9: Download Trained Weights
Run this cell to download the trained `best.pt` file directly to your local computer.

In [ ]:
from google.colab import files
files.download('runs/detect/train/weights/best.pt')